# 🥈 Silver Layer: Real-Time Stream Processing
This notebook implements **Spark Structured Streaming** to ingest raw JSON market data from the Bronze landing zone. It performs schema enforcement and data cleaning before persisting the results into a Delta table for analytical use.

### ⚙️ Transformation Strategy
1. **Schema Enforcement:** Defines a strict `StructType` to interpret the raw JSON payloads accurately.
2. **Streaming Read:** Uses `spark.readStream` to monitor the Lakehouse landing folder for new files.
3. **Data Quality:** Casts data types and adds a standardized `processing_time` and `event_time`.
4. **Delta Sink:** Writes the stream to a managed Delta table with **Checkpoints** to ensure "exactly-once" processing semantics.

In [3]:
import pyspark.sql.functions as f
from pyspark.sql.types import *

# 1. Define the Schema 
schema = StructType([
    StructField("Date", LongType(), True),
    StructField("Open", DoubleType(), True),
    StructField("High", DoubleType(), True),
    StructField("Low", DoubleType(), True),
    StructField("Close", DoubleType(), True),
    StructField("Adj Close", DoubleType(), True),
    StructField("Volume", LongType(), True),
    StructField("ticker", StringType(), True),
    StructField("timestamp", StringType(), True)
])

# 2. Start the Stream Reader
df = (
    spark.readStream
    .schema(schema)
    .option("maxFilesPerTrigger", 1) 
    .json("Files/landing/")
)

# 3. Clean and Transform
silver_df = df.withColumnRenamed("Adj Close", "Adj_Close") \
              .withColumn("processing_time", f.current_timestamp()) \
              .withColumn("event_time", f.to_timestamp("timestamp"))

# 4. Write to Silver Delta Table
query = (
    silver_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "Files/checkpoints/silver_stock")
    .toTable("silver_stock_prices")
)

StatementMeta(, 1641cd08-2c53-4f15-b367-d189da157009, 5, Finished, Available, Finished, False)